# Combo 4 - Chakra training: RESUME notebook

The first run (Tesla T4, ~38 min/epoch) timed out after **19/30 epochs**. The training script only saved `chakra_transformer_best.pth` (best val Dice, last improvement = epoch 17, Dice 0.7854), so this notebook resumes from those weights with the LR schedule fast-forwarded to where it left off.

**Before running:** attach the saved weights as an input (see Step A). **Never** run `rm -rf chakramodel` in this notebook - the repo folder holds `src/weights/`.

Original description: trains a compact MobileNetV3 student from the multi-scale Chakra ViT teacher. The ViT remains the accuracy reference.

In [ ]:
!git clone -q https://github.com/varshasrao58/chakramodel.git /kaggle/working/chakramodel
%cd /kaggle/working/chakramodel
!pip install -q timm==0.9.16 opencv-python-headless
import os, glob, shutil, subprocess
assert os.path.exists('/kaggle/input'), 'Run this notebook on Kaggle with GPU enabled'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())

In [ ]:
import os
import glob
import shutil
from pathlib import Path

# 1. Define target paths explicitly expected by the training script
target_images = Path('/kaggle/working/chakramodel/src/data/kvasir-seg/images')
target_masks = Path('/kaggle/working/chakramodel/src/data/kvasir-seg/masks')

target_images.mkdir(parents=True, exist_ok=True)
target_masks.mkdir(parents=True, exist_ok=True)

# 2. Define the root of your newly attached dataset
dataset_root = Path('/kaggle/input/datasets/gokulrocky/chakra-leakbench-v1')

if not dataset_root.exists():
    raise FileNotFoundError(f"Could not find dataset at {dataset_root}. Check the sidebar.")

# 3. Flatten and copy all images and masks from the nested folders
print("Consolidating datasets into working directory...")
for img_path in dataset_root.rglob('images/*.*'):
    if img_path.is_file():
        shutil.copy(img_path, target_images / img_path.name)
        
for mask_path in dataset_root.rglob('masks/*.*'):
    if mask_path.is_file():
        shutil.copy(mask_path, target_masks / mask_path.name)

# 4. Find the teacher checkpoint (if uploaded)
teacher_matches = glob.glob('/kaggle/input/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pth', recursive=True)
# RESUME: don't mistake the checkpoint we are resuming from for a teacher checkpoint (the original run used ImageNet init)
teacher_matches = [p for p in teacher_matches if 'chakra_transformer' not in os.path.basename(p) and os.path.basename(p) != 'last.pth']
teacher = teacher_matches[0] if teacher_matches else ''

print(f'images={len(list(target_images.glob("*")))}')
print(f'masks={len(list(target_masks.glob("*")))}')
print(f'teacher={teacher or "ImageNet initialization"}')

## Step A - Locate the saved checkpoint
Attach it first: **right sidebar -> Add Input** -> either *your previous notebook's Output* (Your Work tab) or a Dataset you uploaded the `.pth` to. This cell finds it, makes a safety copy outside the repo, and tells you what's inside.

In [ ]:
import glob, os, shutil, torch
from pathlib import Path

CKPT_DIR = Path('/kaggle/working/ckpt'); CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- EDIT IF NEEDED ----------------
CKPT_EPOCH   = 17      # epochs the checkpoint has completed. best.pth was last saved at epoch 17 in the original log.
                       # (If you know it holds epoch-19 weights, use 19 and BEST_DICE = 0.8012.)
BEST_DICE    = 0.7854  # best val Dice so far
TOTAL_EPOCHS = 30      # keep 30: the cosine LR schedule is defined over 30 epochs
RESUME_CKPT  = None    # None = auto-detect (prefers last.pth, else chakra_transformer_best.pth)
# ------------------------------------------------

if RESUME_CKPT is None:
    found = (sorted(glob.glob('/kaggle/input/**/last.pth', recursive=True))
             or sorted(glob.glob('/kaggle/input/**/chakra_transformer_best.pth', recursive=True)))
    assert found, 'No checkpoint found under /kaggle/input - add it via Add Input (Step A) and re-run this cell.'
    RESUME_CKPT = found[0]
print(f'Using: {RESUME_CKPT}  ({os.path.getsize(RESUME_CKPT)/1e6:.0f} MB)')

ck = torch.load(RESUME_CKPT, map_location='cpu')
if isinstance(ck, dict) and 'completed_epochs' in ck:            # a last.pth written by the patched script
    CKPT_EPOCH = int(ck['completed_epochs']); print('last.pth says completed_epochs =', CKPT_EPOCH)
print('checkpoint type:', 'dict with keys ' + str(list(ck)[:6]) if isinstance(ck, dict) else type(ck))
print('contains optimizer state:', isinstance(ck, dict) and 'optimizer' in ck)
del ck
assert 0 <= CKPT_EPOCH < TOTAL_EPOCHS

shutil.copy(RESUME_CKPT, CKPT_DIR / 'resume_source_backup.pth')   # untouched copy, outside the repo
print(f'Will resume at epoch {CKPT_EPOCH+1}/{TOTAL_EPOCHS}  ({TOTAL_EPOCHS-CKPT_EPOCH} epochs left, ~{(TOTAL_EPOCHS-CKPT_EPOCH)*38/60:.1f} h at the original ~38 min/epoch)')

## Step B - Add resume support to `train_transformer.py`
The script has no resume option (it only saves the best weights), so this cell patches it in place: load weights, fast-forward the LR schedule, start the epoch loop at `CKPT_EPOCH`, keep the best-Dice value, and save a full resumable `ckpt/last.pth` after **every** epoch. It is idempotent, checks the result compiles, keeps a `.orig` backup, and if it can't recognise the script's structure it changes nothing and prints the relevant lines.

In [ ]:
SCRIPT_PATH = '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py'
import re, shutil, py_compile, difflib
from pathlib import Path

SCRIPT = Path(SCRIPT_PATH)
BACKUP = Path(str(SCRIPT) + '.orig')
MARK = '# >>> RESUME PATCH'

def stmt_end(lines, i):
    """index of the last line of the (possibly multi-line) statement starting at lines[i]"""
    depth = 0
    for j in range(i, len(lines)):
        depth += sum(lines[j].count(c) for c in '([{') - sum(lines[j].count(c) for c in ')]}')
        if depth <= 0:
            return j
    return i

def block(ind, text):
    return [ind + l if l.strip() else l for l in text.strip('\n').split('\n')]

src = SCRIPT.read_text()
if MARK not in src and not BACKUP.exists():
    shutil.copy(SCRIPT, BACKUP)
if MARK in src:
    print('Script is already patched - nothing to do.')
else:
    lines = src.split('\n')
    problems, notes = [], []

    # 1) optimizer line -> gives us the model + optimizer variable names
    oi = next((i for i, l in enumerate(lines)
               if re.match(r'^\s*(\w+)\s*=\s*[\w\.]*(Adam|AdamW|SGD|RAdam)\w*\(', l)), None)
    if oi is None:
        problems.append('could not find the line that creates the optimizer (expected  opt = ...Adam(model.parameters(), ...))')
    else:
        m = re.match(r'^(\s*)(\w+)\s*=', lines[oi])
        ind, OPT = m.group(1), m.group(2)
        pm = re.search(r'([\w\.]+)\.parameters\(\)', '\n'.join(lines[oi:stmt_end(lines, oi) + 1]))
        if not pm:
            problems.append('optimizer is not built from <model>.parameters() (maybe param groups) - cannot tell which variable is the model')
        else:
            MODEL = pm.group(1)

    # 2) epoch loop
    li = next((i for i, l in enumerate(lines) if re.match(r'^\s*for\s+(\w+)\s+in\s+range\(', l) and 'epoch' in l.lower()), None)
    if li is None:
        problems.append('could not find the epoch loop (expected  for epoch in range(N):)')
    else:
        lm = re.match(r'^(\s*for\s+(\w+)\s+in\s+)range\((.*)\)\s*:\s*$', lines[li])
        if not lm or ',' in lm.group(3):
            problems.append(f'epoch loop is not a simple range(N): {lines[li].strip()!r}')
        else:
            EPV = lm.group(2)

    # 3) epoch summary print line (to attach the per-epoch checkpoint save)
    pi = next((i for i, l in enumerate(lines) if 'Epoch [' in l and 'print' in l), None)
    if pi is None:
        problems.append('could not find the per-epoch print ("Epoch [ ...")')

    if problems:
        print('AUTO-PATCH NOT APPLIED - nothing was changed. Reasons:')
        for p in problems: print('  -', p)
        print('\nRelevant lines of the script (paste these to Claude if you want an exact patch):')
        for i, l in enumerate(lines):
            if re.search(r'optim|scheduler|for .* in range|Epoch \[|best|torch\.save|load_state_dict|DataParallel', l):
                print(f'{i+1:4d}: {l}')
        raise SystemExit
    else:
        edits = []   # (line_index, 'before'|'after'|'replace', new_lines)
        # model load, before optimizer creation
        edits.append((oi, 'before', block(ind, f'''
{MARK} (load weights)
import os as _os, torch as _torch
_rk = {{}}
if _os.environ.get('RESUME_CKPT'):
    _ck = _torch.load(_os.environ['RESUME_CKPT'], map_location='cpu')
    _sd = _ck
    if isinstance(_ck, dict):
        for _k in ('model', 'model_state_dict', 'state_dict'):
            if isinstance(_ck.get(_k), dict):
                _sd = _ck[_k]; break
    _sd = {{(k[7:] if k.startswith('module.') else k): v for k, v in _sd.items()}}
    _tgt = {MODEL}.module if hasattr({MODEL}, 'module') else {MODEL}
    _res = _tgt.load_state_dict(_sd, strict=False)
    print(f"[resume] weights loaded from {{_os.environ['RESUME_CKPT']}} | missing={{len(_res.missing_keys)}} unexpected={{len(_res.unexpected_keys)}}")
    if len(_res.missing_keys) > 0.05 * len(_tgt.state_dict()):
        raise RuntimeError('Checkpoint does not match the model (too many missing keys) - refusing to train from a half-loaded model')
    _rk = _ck if isinstance(_ck, dict) else {{}}
''')))
        oe = stmt_end(lines, oi)
        # optimizer state (only present in last.pth), after optimizer creation
        after_opt = block(ind, f'''
if isinstance(_rk.get('optimizer'), dict):
    _lr0 = [g['lr'] for g in {OPT}.param_groups]
    {OPT}.load_state_dict(_rk['optimizer'])
    for _g, _l in zip({OPT}.param_groups, _lr0): _g['lr'] = _l  # keep base LR so the schedule fast-forward below is exact
    print('[resume] optimizer state restored')
''')
        # scheduler fast-forward
        si = next((i for i, l in enumerate(lines) if re.match(
            r'^\s*(\w+)\s*=\s*[\w\.]*(CosineAnnealingLR|StepLR|MultiStepLR|ExponentialLR|LambdaLR)\(', l)), None)
        if si is not None:
            sm = re.match(r'^(\s*)(\w+)\s*=', lines[si]); SCH = sm.group(2); sind = sm.group(1)
            se = stmt_end(lines, si)
            sched_block = block(sind, f'''
for _ in range(int(_os.environ.get('RESUME_START', '0'))): {SCH}.step()
print('[resume] LR after fast-forwarding schedule:', {OPT}.param_groups[0]['lr'])
''')
            if se == oe: after_opt += sched_block
            else: edits.append((se, 'after', sched_block))
        else:
            notes.append('no epoch-stepped LR scheduler found - LR is probably computed from the epoch inside the loop, which the loop change below handles')
        edits.append((oe, 'after', after_opt))
        # loop start
        lm = re.match(r'^(\s*for\s+(\w+)\s+in\s+)range\((.*)\)\s*:\s*$', lines[li])
        edits.append((li, 'replace', [f"{lm.group(1)}range(int(__import__('os').environ.get('RESUME_START', '0')), {lm.group(3)}):"]))
        # best-metric initial value
        bi = next((i for i, l in enumerate(lines) if re.match(
            r'^\s*best\w*\s*=\s*(0|0\.0|-1|-1\.0|float\([\'"]-inf[\'"]\))\s*$', l)), None)
        if bi is not None:
            bm = re.match(r'^(\s*)(best\w*)\s*=\s*(.*)$', lines[bi])
            edits.append((bi, 'replace', [f"{bm.group(1)}{bm.group(2)} = float(__import__('os').environ.get('RESUME_BEST', {bm.group(3)}))"]))
        else:
            notes.append('best-metric variable not found - a worse epoch could overwrite chakra_transformer_best.pth (a backup is made by the notebook before training)')
        # per-epoch resumable checkpoint, after the epoch print
        pe = stmt_end(lines, pi); pind = re.match(r'^(\s*)', lines[pi]).group(1)
        edits.append((pe, 'after', block(pind, f'''
{MARK} (resumable checkpoint every epoch, outside the repo)
_t = {MODEL}.module if hasattr({MODEL}, 'module') else {MODEL}
__import__('os').makedirs('/kaggle/working/ckpt', exist_ok=True)
__import__('torch').save({{'model': _t.state_dict(), 'optimizer': {OPT}.state_dict(), 'completed_epochs': {EPV} + 1}}, '/kaggle/working/ckpt/last.tmp')
__import__('os').replace('/kaggle/working/ckpt/last.tmp', '/kaggle/working/ckpt/last.pth')
''')))
        # apply from the bottom up so indices stay valid
        _order = {'after': 0, 'replace': 1, 'before': 2}   # same-line ties: after first, before last
        for idx, kind, new in sorted(edits, key=lambda e: (-e[0], _order[e[1]])):
            if kind == 'before':  lines[idx:idx] = new
            elif kind == 'after': lines[idx+1:idx+1] = new
            else:                 lines[idx:idx+1] = new
        SCRIPT.write_text('\n'.join(lines))
        try:
            py_compile.compile(str(SCRIPT), doraise=True)
        except py_compile.PyCompileError as e:
            shutil.copy(BACKUP, SCRIPT)
            raise RuntimeError(f'Patched script has a syntax error, original restored: {e}')
        print('Patched OK. Diff vs original:\n')
        print('\n'.join(difflib.unified_diff(src.split('\n'), lines, 'original', 'patched', lineterm='', n=1)))
        for n in notes: print('NOTE:', n)

## Step C - Resume training
Same arguments as the original run (`--epochs 30 --batch-size 8`). **Check the first lines of output:** you should see `[resume] weights loaded ... missing=0`, and `[resume] LR after fast-forwarding` close to **4.0e-05** (resume from epoch 17) or **3.0e-05** (from epoch 19), then `Epoch [ 18/30]`. If any of that is missing, stop and fix before spending hours.

In [ ]:
%cd /kaggle/working/chakramodel
import subprocess, sys, os

env = os.environ.copy()
env['PYTHONPATH'] = '/kaggle/working/chakramodel:/kaggle/working/chakramodel/src:' + env.get('PYTHONPATH', '')
env['RESUME_CKPT']  = RESUME_CKPT
env['RESUME_START'] = str(CKPT_EPOCH)
env['RESUME_BEST']  = str(BEST_DICE)

# Safety copy of the current best weights if the repo already has them (the script may overwrite on improvement)
best_in_repo = '/kaggle/working/chakramodel/src/weights/chakra_transformer_best.pth'
os.makedirs(os.path.dirname(best_in_repo), exist_ok=True)
if not os.path.exists(best_in_repo):
    shutil.copy(RESUME_CKPT, best_in_repo)     # so 'best' exists even if no epoch beats BEST_DICE

command = [sys.executable, '/kaggle/working/chakramodel/src/chakra_transformer/train_transformer.py',
           '--epochs', str(TOTAL_EPOCHS), '--batch-size', '8']
if teacher: command += ['--teacher-checkpoint', teacher]
subprocess.run(command, check=True, env=env)

## Step D - Export the results
`ckpt/last.pth` (resumable, written every epoch) and the best weights live in `/kaggle/working`, so they show up in the notebook's **Output** after *Save Version*. If this session also times out, repeat from Step A using `last.pth`: it restores weights **and** optimizer state, and the epoch counter is read from it automatically.

In [ ]:
best_in_repo = '/kaggle/working/chakramodel/src/weights/chakra_transformer_best.pth'
if os.path.exists(best_in_repo):
    shutil.copy(best_in_repo, '/kaggle/working/ckpt/chakra_transformer_best.pth')
for p in sorted(Path('/kaggle/working/ckpt').glob('*.pth')):
    print(f'{p.name:40s} {p.stat().st_size/1e6:8.0f} MB')

## Optional - student speed benchmark (unchanged from the original notebook)
Needs `weights/combo4_student_best.pth` from the student-distillation script, which is separate from the ViT run above.

In [ ]:
import torch, time
from src.chakra_transformer.student_segmenter import ChakraLiveStudent
student = ChakraLiveStudent().cuda().eval()
student.load_state_dict(torch.load('weights/combo4_student_best.pth', map_location='cuda'))
x = torch.randn(1, 3, 352, 352, device='cuda')
for _ in range(20): student(x)
torch.cuda.synchronize(); start = time.perf_counter()
for _ in range(100): student(x)
torch.cuda.synchronize(); elapsed = time.perf_counter() - start
print(f'parameters={sum(p.numel() for p in student.parameters()):,}; FPS={100 / elapsed:.2f}; latency_ms={elapsed * 10:.2f}')
shutil.copy('weights/combo4_student_best.pth', '/kaggle/working/combo4_student_best.pth')